# RSNA Knee: упаковка кэша в один файл

Датасет из ~2200 мелких файлов Kaggle не всегда отдаёт кнопкой Download («Data not available. This dataset
may need its archive recreated»). Этот ноутбук берёт одну готовую часть кэша из `03_build_cache_kaggle.ipynb`
и складывает её в **один файл** `rsna_raw_partK.tar`. Один файл в Data Explorer скачивается по прямой ссылке.

Данные не меняются: `.npz` внутри уже сжаты, поэтому tar без сжатия, это просто «коробка» для файлов.

## Как запустить

1. Kaggle -> New Notebook -> File -> Import Notebook -> этот файл.
2. Input: датасет с частью кэша (тот, где лежит `index_part0.csv` или `index_part1.csv`). Можно подключить оба.
3. В конфиге `PART` = номер части, которую упаковываем (как в `index_partK.csv`).
4. Accelerator None -> **Save Version -> Save & Run All** (около 10-15 минут).
5. Output -> **New Dataset**. Для второй части: **Copy & Edit** этого ноутбука, другой `PART`, то же самое
   (из одного ноутбука Kaggle разрешает только один датасет).

## Как скачать и распаковать на Mac

Data Explorer -> `rsna_raw_partK.tar` -> скачать. Если браузер открывает прямую ссылку, её можно отдать в `curl`
и распаковывать прямо по ходу загрузки, не храня tar на диске:

```bash
curl -L "<ссылка>" | tar -x -C rsna-knee-abnormality-detection/data/raw
```

In [ ]:
import json
import tarfile
import time
from pathlib import Path

PART = 0                                   # какую часть упаковываем: 0 или 1 (как в index_partK.csv)
INPUT = Path("/kaggle/input")
OUT_DIR = Path("/kaggle/working")
T0 = time.time()


def log(msg):
    print(f"[{time.time() - T0:6.1f}s] {msg}", flush=True)

## Находим часть кэша и проверяем, всё ли на месте

Ищем `index_partK.csv` среди подключённых датасетов. Рядом с ним лежат `studies/`, `rsna_raw_meta.json` и `train.csv`.
Число файлов в `studies/` сверяется с индексом: если чего-то не хватает, ноутбук остановится до упаковки.

In [ ]:
import os

# ищем во всех подключённых датасетах (os.walk заходит и в смонтированные, и в связанные папки)
found = sorted(Path(root) / f for root, _, fs in os.walk(INPUT, followlinks=True)
               for f in fs if f.startswith("index_part") and f.endswith(".csv"))
print("подключено в Input:", sorted(p.name for p in INPUT.iterdir()) if INPUT.is_dir() else "ничего")
print("найдены части кэша:", [f"{p.name} в {p.parent}" for p in found] or "ни одной")
hits = [p for p in found if p.name == f"index_part{PART}.csv"]
assert hits, (f"не найден index_part{PART}.csv. Поставьте PART равным номеру из найденных выше "
              f"index_partK.csv или подключите датасет с нужной частью")
ROOT = hits[0].parent
log(f"часть {PART} найдена в {ROOT}")

import pandas as pd
index = pd.read_csv(ROOT / f"index_part{PART}.csv")
expected = sorted(index["StudyInstanceUID"].astype(str).unique())
files = sorted((ROOT / "studies").glob("*.npz"))
have = {f.stem for f in files}
missing = [s for s in expected if s not in have]
size_gb = sum(f.stat().st_size for f in files) / 1024 ** 3
log(f"в индексе {len(expected)} исследований, в studies/ {len(files)} файлов, {size_gb:.2f} ГБ, не хватает {len(missing)}")
assert not missing, f"в датасете нет {len(missing)} файлов (например {missing[:3]}): часть собрана не полностью"

## Упаковка

В архив кладутся индекс, настройки сборки, `train.csv` и все файлы исследований. Пути внутри архива такие же,
как должны лежать локально: `studies/<ID>.npz`, `index_partK.csv` и т.д.

In [ ]:
out = OUT_DIR / f"rsna_raw_part{PART}.tar"
with tarfile.open(out, "w") as tar:                     # без сжатия: npz уже сжаты
    for name in (f"index_part{PART}.csv", "rsna_raw_meta.json", "train.csv"):
        if (ROOT / name).is_file():
            tar.add(ROOT / name, arcname=name)
    for i, f in enumerate(files, 1):
        tar.add(f, arcname=f"studies/{f.name}")
        if i % 500 == 0 or i == len(files):
            log(f"  упаковано {i}/{len(files)}")
log(f"готово: {out.name}, {out.stat().st_size / 1024 ** 3:.2f} ГБ")

with tarfile.open(out) as tar:                          # проверка: в архиве все файлы
    n = sum(1 for m in tar.getmembers() if m.name.startswith("studies/"))
assert n == len(files), (n, len(files))
log(f"проверка архива: {n} исследований внутри")